# 🔗 chain_client_explore — the chain, from Python (skeleton v1)

Until M1.5, Python knew the chain only as a **cardboard prop** (`FakeChain`, a dict), and the
real contracts spoke only to `forge`/`cast`. This notebook is the moment they meet: the
`ChainClient` — the project's **first real adapter** — signs offers with real keys, submits
them to the real `A2ASettlement` on a live Anvil, and reads tickets back as pydantic models.

The one idea to hold onto: **the client fills the exact hole the fake fills** — both satisfy
the `EntitlementReader` Protocol — so the controller wiring built in M0.3 accepts this real
adapter *unchanged*. That is what "skeleton v1" means.

**Prerequisites:** `anvil` on PATH and built artifacts (`cd contracts && forge build`, done
automatically if you ran any forge command). Runs fully self-contained — it launches its own
throwaway chain.

**Companions:** [`docs/03-interfaces.md`](../../docs/03-interfaces.md) §2/§4 ·
[`contracts/EXPLORE-fulfill.md`](../../contracts/EXPLORE-fulfill.md) (the same purchase, by
hand with `cast`) · `chainmcp/tests/test_cross_stack.py` (these cells, as pinned tests).

## 1. A throwaway chain, frozen at story time

`launch_anvil` starts a disposable Anvil and deploys **MockTOK first, then A2ASettlement** —
the same order as `script/Deploy.s.sol`, so the token lands at the address
`fixtures.MOCK_TOK` promised (a fresh chain's addresses depend only on deployer + nonce).

We pin the clock to ~13:32 *story time* (15 Sep 2025) because the canonical offer's quote
expires at 14:20 — and **chain time is the only clock that counts** (ADR-004). Your wall
clock is past that date; the chain's isn't, so the offer is alive here.

In [1]:
from a2a_interfaces.fixtures import WINDOW, MOCK_TOK

from chainmcp.testing import launch_anvil

STORY_TIME = WINDOW.start - 1680  # 13:32 — 28 min before Ada's window opens

anvil = launch_anvil(timestamp=STORY_TIME)
print("rpc:       ", anvil.rpc_url)
print("deployment:", anvil.deployment)
assert anvil.deployment["MockTOK"] == MOCK_TOK  # the fixture's promise, kept


rpc:        http://127.0.0.1:39815
deployment: {'v': 0, 'chainId': 31337, 'MockTOK': '0x5FbDB2315678afecb367f032d93F642f64180aa3', 'A2ASettlement': '0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512'}


## 2. Identity = client (the custody rule)

`chainmcp` is **the only package that ever holds a private key** (hard rule 2). The rule has
a shape: one `ChainClient` per identity, constructed *with* that identity's key, which never
leaves the object — everyone else sees addresses and signatures.

The keys below are Anvil's well-known dev accounts (public constants, not secrets):
account #0 *is* Ada, #1 *is* Bell — the same cast as every test and lab so far.

In [2]:
from a2a_interfaces import EntitlementReader
from chainmcp import ChainClient
from chainmcp.testing import ANVIL_KEYS

make = lambda key: ChainClient(anvil.rpc_url, key, deployment=anvil.deployment, poll_interval=0.2)
ada, bell, carol = make(ANVIL_KEYS["ada"]), make(ANVIL_KEYS["bell"]), make(ANVIL_KEYS["carol"])

print("Ada  =", ada.address)
print("Bell =", bell.address)

# Rule 7, live: the REAL adapter fits the same hole as FakeChain.
assert isinstance(ada, EntitlementReader)
print("ChainClient satisfies EntitlementReader ✓  (same Protocol as the fake)")


Ada  = 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Bell = 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
ChainClient satisfies EntitlementReader ✓  (same Protocol as the fake)


## 3. The clock and the money

`chain_time()` reads the latest block's timestamp — this number, and nothing else, decides
offer expiry today and session expiry later (M4.5). Then stage money: the faucet is open
because TOK is lab currency; what matters is the *mechanics* (allowance → pull), not scarcity.

In [3]:
import datetime

t = ada.chain_time()
print("chain_time:", t, "=", datetime.datetime.fromtimestamp(t, datetime.UTC).strftime("%H:%M %d %b %Y"))

ada.faucet(50 * 10**18)          # Ada: buying money
carol.faucet(60 * 10**18)        # Carol: will buy Bell's first six tickets (watch §5)

fmt = lambda wei: f"{wei / 10**18:g} TOK"
for name, who in [("Ada", ada.address), ("Bell", bell.address), ("Carol", carol.address)]:
    print(f"{name:6} {fmt(ada.tok_balance(who))}")


chain_time: 1757943120 = 13:32 15 Sep 2025
Ada    50 TOK
Bell   0 TOK
Carol  60 TOK


## 4. The offer, and the hash two languages must agree on

The canonical offer (Ada's 50 Mbps, 10 TOK, ticket-to-be #7) comes from
`a2a_interfaces.fixtures` — the one source of truth. On the wire it's **camelCase**, because
this JSON mirrors the Solidity EIP-712 struct field-for-field: *what is signed must equal
what the contract verifies.*

`offer_digest` builds the EIP-712 hash in Python: struct-hash the twelve fields (dynamic
`bytes` enter as their keccak — the rule that bites everyone), bind them to the **domain**
`("A2AProvisioning", "0", 31337, settlement)`, and keccak the result. The contract's
`hashOffer` does the same in Solidity. **If they ever disagree by one byte, every signature
recovers to a stranger** — so we check, every time, against the live contract:

In [4]:
from a2a_interfaces.fixtures import CANONICAL_OFFER

CANONICAL_OFFER.model_dump(by_alias=True)  # the wire shape — read it aloud in story terms


{'provider': '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
 'consumer': '0x0000000000000000000000000000000000000000',
 'serviceType': 0,
 'resourceId': '0x0000000000000000000000000000000000000000000000000000000000000007',
 'params': '0x0000000000000000000000000000000000000000000000000000000002faf0800000000000000000000000000000000000000000000000000000000000000001',
 'startTime': 1757944800,
 'endTime': 1757952000,
 'paymentToken': '0x5FbDB2315678afecb367f032d93F642f64180aa3',
 'price': '10000000000000000000',
 'validUntil': 1757946000,
 'salt': '0x0000000000000000000000000000000000000000000000000000000000005a17',
 'termsHash': '0x2222222222222222222222222222222222222222222222222222222222222222'}

In [5]:
python_digest = ada.offer_digest(CANONICAL_OFFER)
contract_digest = ada._settlement.functions.hashOffer(ada._offer_tuple(CANONICAL_OFFER)).call()

print("python:  0x" + python_digest.hex())
print("solidity:0x" + contract_digest.hex())
assert python_digest == contract_digest
print("\nTwo languages, one hash — the cross-stack seam is sound. ✓")


python:  0xbc4955e34d4f9670e5fe370dab1be118b8bba1ce7f4b643622d902945cf3d9a6
solidity:0xbc4955e34d4f9670e5fe370dab1be118b8bba1ce7f4b643622d902945cf3d9a6

Two languages, one hash — the cross-stack seam is sound. ✓


## 5. Bell signs — and six tickets sell before Ada's

`sign_offer` is **off-chain**: no transaction, no gas; the chain has no idea Bell promised
anything. The 65-byte signature is the promise.

Then a bit of stage-setting that makes the story literally true: in the story, Ada's ticket
is **#7** because Bell sold six before hers. Here Carol buys those six (same terms, six fresh
salts — each salt is a new ticket-stub serial needing a new signature).

In [6]:
from eth_account import Account

signed = bell.sign_offer(CANONICAL_OFFER, terms_doc={"sla": {"latency_ms": 20}})
print("signature:", signed.signature[:24], "…", signed.signature[-8:])

# local sanity check (the contract does the same recover in Solidity):
from chainmcp import encode_offer
recovered = Account.recover_message(
    encode_offer(CANONICAL_OFFER, anvil.deployment["chainId"], anvil.deployment["A2ASettlement"]),
    signature=signed.signature,
)
print("recovers to:", recovered, "== Bell ✓" if recovered == bell.address else "≠ Bell ✗")


signature: 0x892842e5b351d82e38b391 … b969861b
recovers to: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 == Bell ✓


In [7]:
for i in range(1, 7):
    pre_sale = CANONICAL_OFFER.model_copy(update={"salt": "0x" + f"{i:064x}"})
    _, eid = carol.approve_and_fulfill(bell.sign_offer(pre_sale))
    print(f"ticket #{eid} → Carol   (salt …{i:04x})")


ticket #1 → Carol   (salt …0001)
ticket #2 → Carol   (salt …0002)
ticket #3 → Carol   (salt …0003)
ticket #4 → Carol   (salt …0004)


ticket #5 → Carol   (salt …0005)


ticket #6 → Carol   (salt …0006)


## 6. The purchase: money and ticket in the same breath

Two steps because ERC-20 payment is *pull*-based: Ada first grants an **allowance** (exactly
10 TOK), then `fulfill` pulls the payment and mints the ticket — one transaction, so either
everything happens or nothing does (invariant I3). `approve_and_fulfill` wraps both.

In [8]:
before = {"ada": ada.tok_balance(ada.address), "bell": ada.tok_balance(bell.address)}

tx_hash, ticket_id = ada.approve_and_fulfill(signed)

print(f"ticket #{ticket_id} minted in tx {tx_hash[:18]}…")
print("owner_of(7):", ada.owner_of(ticket_id), "(Ada)")
print(f"Ada  paid    {fmt(before['ada'] - ada.tok_balance(ada.address))}")
print(f"Bell earned  {fmt(ada.tok_balance(bell.address) - before['bell'])}")
print("salt punched:", ada.offer_consumed(CANONICAL_OFFER))
assert ticket_id == 7


ticket #7 minted in tx 0x32ec78a5a9083054…
owner_of(7): 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 (Ada)
Ada  paid    10 TOK
Bell earned  10 TOK
salt punched: True


## 7. Reading the ticket back — as a typed, validated model

`get()` returns an `EntitlementView` (docs/03 §4.1): the eight on-chain fields with the
`params` blob **ABI-decoded** into a real `BandwidthParams`. This is the exact object the
controller's predicate will judge in M4.1 — no web3 types leak past this point.

In [9]:
view = ada.get(7)
print(view.model_dump_json(indent=2))
assert view.params.capacity_bps == 50_000_000 and view.issuer == bell.address


{
  "id": 7,
  "issuer": "0x70997970C51812dc3A010C7d01b50e0d17dc79C8",
  "service_type": 0,
  "resource_id": "0000000000000000000000000000000000000000000000000000000000000007",
  "params": {
    "capacity_bps": 50000000,
    "qos_class": 1
  },
  "start_time": 1757944800,
  "end_time": 1757952000,
  "revoked": false,
  "terms_hash": "2222222222222222222222222222222222222222222222222222222222222222"
}


## 8. Cheating from Python: replay, then tamper

The same four refusals you met in the cast lab, now as `ChainRevert` exceptions. Note the
names: the contract's custom errors travel through the ABI decoder into Python — the same
names `FakeChain` raises, so the e2e suite maps one onto the other one-for-one.

In [10]:
from chainmcp import ChainRevert

try:
    ada.approve_and_fulfill(signed)                    # cheat #1: redeem twice
except ChainRevert as err:
    print("replay      →", err.name)

cheaper = signed.offer.model_copy(update={"price": str(10**18)})  # cheat #2: 1 TOK "discount"
try:
    ada.approve_and_fulfill(signed.model_copy(update={"offer": cheaper}))
except ChainRevert as err:
    print("tampered    →", err.name)


replay      → OfferAlreadyUsed
tampered    → BadSignature


## 9. The watcher: how revocation will kill live sessions

`watch_revoked(callback)` polls the chain for `Revoked(id)` events from a background thread —
the mechanism that lets the controller tear down a session **mid-window** the moment Bell
pulls the flag (M4.5's showpiece). Watch it fire:

In [11]:
import time

seen = []
ada.watch_revoked(seen.append)
time.sleep(0.3)                      # let the watcher note its starting block

bell.revoke(7)                       # Bell pulls the kill switch on-chain

deadline = time.monotonic() + 5
while not seen and time.monotonic() < deadline:
    time.sleep(0.05)
print("watcher delivered:", seen)
print("revoked flag now:", ada.get(7).revoked, "— owner still", ada.owner_of(7)[:8] + "… (no burn, I5)")


watcher delivered: [7]
revoked flag now: True — owner still 0xf39Fd6… (no burn, I5)


## 10. The other signature: the activation proof (EIP-191)

Buying proved *payment*; activating must prove *ownership, live, to the controller* — done
by signing the docs/03 §3.2 challenge string with a plain `personal_sign` (EIP-191, simpler
than EIP-712: no struct, just a tagged string). The controller (M4.2) will recover this and
compare against `ownerOf`. No Solidity involved — this proof is verified in Python.

In [12]:
from eth_account.messages import encode_defunct

signature, address = ada.sign_activation_proof(
    entitlement_id=7, nonce="0xabcd", controller_id="bw-ctrl-1", expires_at=WINDOW.start + 300
)
message = f"a2a-activate|bw-ctrl-1|0xabcd|7|{WINDOW.start + 300}"
print("message:  ", message)
print("recovers →", Account.recover_message(encode_defunct(text=message), signature=signature))


message:   a2a-activate|bw-ctrl-1|0xabcd|7|1757945100
recovers → 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266


## 11. And skeleton v1 itself

Everything above is exactly what `SKELETON_PROFILE=chain uv run pytest e2e/` wires up: the
same lifecycle script that ran on cardboard now runs here — `ChainWorld` swaps `FakeChain`
for these clients and nothing else changes. The controller can't tell, **by design**.

## What you learned (and where it goes)

| You did | The concept | Next |
|---|---|---|
| `launch_anvil` + deterministic deploy | the §2.4 deployment artifact | `just up` (M6.1) |
| one client per key | custody rule 2, identity = client | per-agent MCP servers (M5.4) |
| digest == `hashOffer` | the cross-stack seam, checked live | it's a pinned CI test now |
| `sign_offer` off-chain | a signature IS the promise | provider agent quotes (M5.3) |
| `approve_and_fulfill` | pull-payment + atomic settle | the consumer graph's `settle` (M5.2) |
| `get()` → `EntitlementView` | ABI blob → typed model at the border | the predicate's input (M4.1) |
| `watch_revoked` fired | event-driven teardown | the revocation showpiece (M4.5) |
| `sign_activation_proof` | EIP-191 vs EIP-712 — two proof kinds | challenge-response auth (M4.2) |

## Experiments to try

- In `chainmcp/src/chainmcp/signing.py`, change the domain `version` to `"1"`, restart the
  kernel, rerun. Watch §4's digest-comparison cell **fail its assert** — that's the M1.5 classic bug, caught by the
  digest check before it could even become a confusing `BadSignature`. Change it back.
- Sign with Carol's client but keep `provider = Bell` in the offer — predict the error name
  before you run it.
- `ada.get(3)` — Carol's ticket #3 is a perfectly readable `EntitlementView`; ownership and
  readability are separate things (I8).
- Call `ada.owner_of(99)` — why `KeyError` and not a revert exception? (Port parity: the
  fake's callers expect dict semantics; see the client docstring.)

In [13]:
# cleanup: stop watcher threads and the throwaway chain
for client in (ada, bell, carol):
    client.close()
anvil.stop()
print("world folded away — rerun from the top anytime")


world folded away — rerun from the top anytime
